# Fraud Sentinel — Relational Data Wrangler

1. Clean / merge / sanitize (all string columns — no `notes` column in CSV)
2. Rule features + weak labels → `predictions_rules.json`
3. **Inference by default** on Qwen2.5-1.5B (strict JSON prompt)
4. Optional LoRA retrain via `RETRAIN=True` / `--retrain`
5. Greedy JSON parse + rule fallback → `predictions.json`

In [ ]:
# Config
import sys
from pathlib import Path

DATA_DIR = Path('.').resolve()
RANDOM_SEED = 42
RETRAIN = False  # set True only when you want to fine-tune LoRA again

print('cwd:', DATA_DIR)
print('python:', sys.executable)
print('RETRAIN:', RETRAIN)

In [ ]:
import fraud_sentinel as fs

fs.DATA_DIR = DATA_DIR
fs.RANDOM_SEED = RANDOM_SEED
fs.RETRAIN = RETRAIN
fs.RULES_OUT = DATA_DIR / 'predictions_rules.json'
fs.FINAL_OUT = DATA_DIR / 'predictions.json'
fs.ADAPTER_DIR = DATA_DIR / 'models' / 'fraud_sentinel_lora'

merged = fs.run_rules_only()
merged[['transaction_id', 'amount', 'merchant_name', 'rule_score', 'weak_is_fraud', 'template_justification']].head(10)

In [ ]:
print('rows:', len(merged))
print('fraud rate:', float(merged['weak_is_fraud'].mean()))
print('injection flags:', int(merged['injection_attempt'].sum()))
print('orphan accounts:', int(merged['orphan_account'].sum()))
print('adapter weights:', fs.adapter_weights_exist())
print('rules file:', fs.RULES_OUT.exists())

In [ ]:
# Inference only by default (uses LoRA if adapter weights exist, else base Qwen)
# Set RETRAIN=True above to fine-tune first.
preds = fs.train_and_infer(merged, retrain=RETRAIN)
print('final predictions:', len(preds))
preds[:3]

In [ ]:
required = {'transaction_id', 'is_fraud', 'confidence', 'justification'}
bad = []
for p in preds:
    if set(p) != required:
        bad.append(('keys', p.get('transaction_id'), set(p)))
    if not isinstance(p['is_fraud'], bool):
        bad.append(('is_fraud_type', p['transaction_id']))
    if not (0.0 <= float(p['confidence']) <= 1.0):
        bad.append(('confidence', p['transaction_id'], p['confidence']))
    if not str(p['justification']).strip():
        bad.append(('empty_justification', p['transaction_id']))

print('schema issues:', len(bad))
print('coverage vs merged:', len(preds), len(merged))
print('wrote:', fs.FINAL_OUT)